## January Model Test on May Images

The objective of this script is to test a YOLO model trained on January images on new sets of May test images

First, let's import the necessary libraries.

In [1]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.8 MB/s eta 0:00:00


In [2]:
import os
import yaml
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### 1. Testing Preparation

We first mount our drive to point to the folder where the testing images are located.

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


Next, we load the trained model

<div class="alert alert-block alert-info">
    
<b>Note:</b> YOLOv8 automatically saves the model on training. The saved model can be found in this path where the training script is located. *runs/detect/train/exp*/weights/*

The model is automatically named as *"best.pt"*

</div>

In [9]:
model_path = os.path.join("/content/drive/MyDrive/Drone_images/JAN_Training/runs/detect/train/weights", "best.pt")
jan_model = YOLO(model_path)

Next we define the path of the test images including the labels (Pascal VOC .xml labels and .txt class files)

In [10]:
test_images = "/content/drive/MyDrive/Drone_images/MAY_Training/images/test"   # folder with test images
test_labels = "/content/drive/MyDrive/Drone_images/MAY_Training/labels/test"   # folder with YOLO .txt labels
xml_labels  = "/content/drive/MyDrive/Drone_images/MAY_Training/labels/test"   # folder with XML files

### 2. Testing the LWIR Images

We make predictions using the trained model

In [11]:
pred_results = jan_model.predict(source = test_images, imgsz = 640, save = True)


image 1/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_0_lwir_113.jpg: 512x640 3 ap_metals, 2 at_plastics, 6.8ms
image 2/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_0_lwir_13.jpg: 512x640 1 ap_plastic, 2 at_plastics, 6.1ms
image 3/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_0_lwir_32.jpg: 512x640 2 ap_metals, 5 at_plastics, 6.2ms
image 4/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_0_lwir_4.jpg: 512x640 1 at_plastic, 6.1ms
image 5/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_0_lwir_64.jpg: 512x640 12 at_plastics, 8.8ms
image 6/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_1_lwir_110.jpg: 512x640 1 ap_plastic, 6 at_plastics, 8.1ms
image 7/905 /content/drive/MyDrive/Drone_images/MAY_Training/images/test/may_afternoon_0_1_lwir_116.jpg: 512x640 3 ap_plastics, 9 at_plastics, 10.

### 3. Model Evaluation

First, we create aand save structured configuration YAML file that will help us in evaluating the model.

In [14]:
class_names = ['ap_metal', 'ap_plastic', 'at_metal', 'at_plastic']

In [19]:
data = {
    "path":"/content/drive/MyDrive/Drone_images/MAY_Training",
    "train": os.path.join("/content/drive/MyDrive/Drone_images/MAY_Training/images/train"),
    "val": os.path.join("/content/drive/MyDrive/Drone_images/MAY_Training/images/val"),
    "test": os.path.join("/content/drive/MyDrive/Drone_images/MAY_Training/images/test"),
    "names": class_names,
}

yaml_dir = "/content/drive/MyDrive/Drone_images/JAN_Testing"
yaml_path = os.path.join(yaml_dir, "eval.yaml")
with open(yaml_path, "w") as f:

    yaml.dump(data, f, default_flow_style = False)

We finally evaluate the performance of our model by printing and saving the precision, recall, 50% and 90% mean Average Precision (mAP) scores.

In [20]:
results = jan_model.val(data = yaml_path, split = "test",
    imgsz = 640, batch = 16, save_json = True, plots = True)

Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 75.6±18.2 MB/s, size: 167.0 KB)
val: Scanning /content/drive/MyDrive/Drone_images/MAY_Training/labels/test.cache... 904 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 905/905 1.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 3.3it/s 17.1s
                   all        905       8611      0.322       0.21      0.193      0.107
              ap_metal        582       1009      0.175     0.0634     0.0417     0.0162
            ap_plastic        592       1330      0.161     0.0654     0.0428     0.0188
              at_metal        671        993      0.238      0.116      0.103     0.0609
            at_plastic        868       5279      0.716      0.594      0.586      0.334
Speed: 0.7ms preprocess, 3.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving /content/ru

We save the model evaluation results for future use.

In [21]:
!cp -r /content/runs/detect/val /content/drive/MyDrive/Drone_images/JAN_Testing/jan_model_predictions_on_may/

And then extract the individual metrics before converting to dataframe and eventually saving the results as a csv file.

In [22]:
precision = results.box.p
recall = results.box.r
ap50 = results.box.ap50
ap5095 = results.box.ap
class_names = results.names

In [23]:
df = pd.DataFrame({
    "class": [class_names[i] for i in range(len(precision))],
    "precision": precision,
    "recall": recall,
    "map50": ap50,
    "map50_95": ap5095,
    "model": ["Jan on May Images"] * len(precision)
    })

RESULTS = "/content/drive/MyDrive/Drone_images/JAN_Testing"
output_file = os.path.join(RESULTS, "jan_model_on_may_images_metrics.csv")
df.to_csv(output_file, index=False)